In [0]:
import pandas as pd
import numpy as np

In [0]:
spark.sql("USE CATALOG 'proyecto_final_prueba'")

In [0]:
catalog = spark.sql("SELECT current_catalog()").first()[0]
schema = 'silver'
table = 'weather'

In [0]:
spark.sql(f"create schema if not exists {catalog}.{schema}")

In [0]:
spark.sql(f"drop table if exists {catalog}.{schema}.{table}")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{schema}.{table} (
  fecha_hora_local TIMESTAMP,
  latitude DOUBLE,
  longitude DOUBLE,
  temperature DOUBLE,
  humidity INT,
  wind_speed DOUBLE,
  weather_code INT,
  weather_desc STRING
)
""")

In [0]:
df = spark.table(f"{catalog}.bronze.{table}").toPandas()
df

In [0]:
columns_list =['hourly.time', 'hourly.temperature_2m', 'hourly.relative_humidity_2m', 'hourly.wind_speed_10m', 'hourly.weather_code']
df_silver = df.explode(columns_list)
df_silver

In [0]:
rename_col = {'hourly.time': 'time',
    'hourly.temperature_2m': 'temperature',
    'hourly.relative_humidity_2m': 'humidity',
    'hourly.wind_speed_10m': 'wind_speed',
    'hourly.weather_code': 'weather_code'}

df_silver = df_silver.rename(columns=rename_col)
df_silver

In [0]:
df_silver['temperature'] = df_silver['temperature'].astype(float)
df_silver['wind_speed'] = df_silver['wind_speed'].astype(float)
df_silver['humidity'] = df_silver['humidity'].astype(int)
df_silver['weather_code'] = df_silver['weather_code'].astype(int)

df_silver['fecha_hora_utc'] = pd.to_datetime(df_silver['time'])
df_silver['fecha_hora_local'] = df_silver['fecha_hora_utc'] - pd.Timedelta(hours=5)
df_silver

In [0]:
condiciones = [
    (df_silver['weather_code'] == 0),
    (df_silver['weather_code'].between(1, 3)),
    (df_silver['weather_code'].between(51, 69))
]
valores = ['Despejado', 'Nublado', 'Lluvia ligera/moderada']
df_silver['weather_desc'] = np.select(condiciones, valores, default='Otro / Extremo')
df_silver

In [0]:

df_silver = df_silver.dropna(subset=['temperature'])

columnas_finales = [
    'fecha_hora_local', 'latitude', 'longitude', 'temperature', 
    'humidity', 'wind_speed', 'weather_code', 'weather_desc'
]
df_silver = df_silver[columnas_finales]
df_silver

In [0]:
df_spark = spark.createDataFrame(df_silver)
df_spark.display()

In [0]:
df_spark.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{schema}.{table}")